# BƯỚC 1: KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EXPLORATORY DATA ANALYSIS - EDA)

## Mục tiêu theo đề bài Practice Exercise 3:
1. Đọc và kiểm tra cấu trúc tập dữ liệu khách hàng (`data/raw/marketing_campaign.csv`, định dạng phân tách bằng tab `\t`).
2. Tạo các đặc trưng phân cụm cốt lõi (`Age = 2014 - Year_Birth`, `Total_Spending = sum(6 cột Mnt*)`), lọc lỗi miền bất hợp lý (`Year_Birth < 1920`, `Income > 200,000`) và chia 80% Train / 20% Test trước khi điền trung vị `Income`.
3. Phân tích giá trị khuyết thiếu (Missing values), phát hiện dị biệt (Outliers) bằng phương pháp IQR (Tukey's Fences).
4. Khảo sát hình thái phân phối, quan hệ song biến và biến nhân khẩu học thật (`Education`, `Marital_Status`).

In [ ]:
# Thêm đường dẫn gốc của dự án để import module từ src/
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import NUMERICAL_FEATURES, PROCESSED_DATA_DIR, RAW_DATA_PATH
from src.data.loader import load_raw_data, split_data, validate_schema
from src.data.preprocessing import (
    add_derived_features,
    audit_missing_and_outliers,
    clean_data,
    compute_train_medians,
    filter_invalid_records,
)
from src.evaluation.visualization import plot_eda_correlations, plot_eda_distributions

In [ ]:
# Nạp dữ liệu gốc và kiểm định tính toàn vẹn schema
raw_df = load_raw_data(RAW_DATA_PATH)
schema_info = validate_schema(raw_df)
raw_df.head()

In [ ]:
# Kiểm toán giá trị khuyết, ngoại lai (IQR) và dị biệt hạng mục trên dữ liệu gốc trước khi làm sạch
audit_goc = audit_missing_and_outliers(raw_df)
print("Giá trị Marital_Status thô:", raw_df["Marital_Status"].value_counts().to_dict())
print("Năm sinh < 1920:", raw_df[raw_df["Year_Birth"] < 1920][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
print("Thu nhập > 200,000:", raw_df[raw_df["Income"] > 200000][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
audit_goc

In [ ]:
# Lọc bản ghi bất hợp lý, chia 80% Train / 20% Test rồi mới điền trung vị Income của tập Train
valid_df = add_derived_features(filter_invalid_records(raw_df))
train_raw, test_raw = split_data(valid_df, save_files=False)
train_medians = compute_train_medians(train_raw)

train_df = clean_data(train_raw, fill_medians=train_medians, drop_invalid_outliers=False)
test_df = clean_data(test_raw, fill_medians=train_medians, drop_invalid_outliers=False)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)
pd.concat([train_df, test_df], ignore_index=True).to_csv(
    PROCESSED_DATA_DIR / "full_customers.csv", index=False
)

print("Thống kê mô tả trên tập Train (80%):")
train_df[NUMERICAL_FEATURES].describe().T

In [ ]:
# Trực quan hóa phân phối, Boxplot phát hiện ngoại lai và tương quan song biến trên tập Train
plot_eda_distributions(train_df)
plot_eda_correlations(train_df)